# 02 · Preprocesamiento — scikit-learn

**Autores:** Santiago Hurtado, Juan Marín, Andrés Parejo

Selección de variables → codificación → escalado → split 80/20
estratificado, exactamente en ese orden (el orden que pide el enunciado).
Se mide el tiempo de cada etapa con `time.time()` para la comparación
final contra PySpark en `07_comparison.ipynb`.

## Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import utils
import time
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_parquet(utils.CLEAN_PARQUET_DRIVE)
print(df.shape)

(2260668, 30)


## 1. Selección de variables relevantes

Ya se seleccionaron y limpiaron en `00_setup_datos.ipynb` (`utils.NUMERIC_COLS`
+ `utils.CATEGORICAL_COLS`, 24 variables + target). Aquí solo se arma la
matriz de features.

In [2]:
print("Numéricas:", utils.NUMERIC_COLS)
print("Categóricas:", utils.CATEGORICAL_COLS)

X_cat_raw = df[utils.CATEGORICAL_COLS].astype(str)
X_num_raw = df[utils.NUMERIC_COLS].astype("float64")
y = df[utils.TARGET_COL].to_numpy()

Numéricas: ['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti', 'fico_range_low', 'fico_range_high', 'emp_length_num', 'revol_util', 'revol_bal', 'open_acc', 'total_acc', 'mort_acc', 'pub_rec', 'delinq_2yrs']
Categóricas: ['term', 'grade', 'purpose', 'home_ownership', 'verification_status', 'addr_state', 'initial_list_status', 'application_type']


## 2. Codificación de variables categóricas

Se usa `OneHotEncoder` (no `LabelEncoder`): las categóricas del proyecto
(`purpose`, `addr_state`, `home_ownership`, ...) son **nominales**, sin
orden natural — `LabelEncoder` les impondría un orden numérico arbitrario
(p.ej. "car" < "credit_card") que un `RandomForest` interpretaría como
información real. `handle_unknown="ignore"` es una red de seguridad para
categorías que no aparezcan en train (con 2.26M filas y categorías raras ya
agrupadas, no debería ocurrir, pero no cuesta nada tenerlo).

In [3]:
t0 = time.time()
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
X_cat_encoded = ohe.fit_transform(X_cat_raw)
cat_feature_names = ohe.get_feature_names_out(utils.CATEGORICAL_COLS).tolist()
t_encode = time.time() - t0
print(f"One-hot: {X_cat_encoded.shape[1]} columnas generadas en {t_encode:.2f}s")

One-hot: 57 columnas generadas en 3.02s


## 3. Escalado con StandardScaler

Necesario porque `RandomForest` no lo requiere estrictamente (es invariante
a escala), pero se aplica porque (a) lo pide el enunciado y (b) LIME, en el
notebook 06, perturba en el espacio de features tal como se lo pasa al
modelo — mantener todas las variables en una escala comparable hace esas
perturbaciones más interpretables.

In [4]:
t0 = time.time()
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num_raw).astype(np.float32)
t_scale = time.time() - t0
print(f"Escalado de {X_num_scaled.shape[1]} columnas numéricas en {t_scale:.2f}s")

Escalado de 15 columnas numéricas en 0.52s


In [5]:
X = np.hstack([X_num_scaled, X_cat_encoded])
feature_names = utils.NUMERIC_COLS + cat_feature_names
print("Matriz final X:", X.shape, "| features:", len(feature_names))

Matriz final X: (2260668, 72) | features: 72


## 4. División train/test 80/20 estratificada

In [6]:
t0 = time.time()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
t_split = time.time() - t0

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Proporción de default en train:", y_train.mean().round(4), "| en test:", y_test.mean().round(4))

Train: (1808534, 72) | Test: (452134, 72)
Proporción de default en train: 0.1188 | en test: 0.1188


## 5. Guardar artefactos para modelado / LIME

In [7]:
t_total = t_encode + t_scale + t_split

joblib.dump(X_train, f"{utils.DATA_DIR}/sklearn_X_train.joblib")
joblib.dump(X_test, f"{utils.DATA_DIR}/sklearn_X_test.joblib")
joblib.dump(y_train, f"{utils.DATA_DIR}/sklearn_y_train.joblib")
joblib.dump(y_test, f"{utils.DATA_DIR}/sklearn_y_test.joblib")
joblib.dump(feature_names, f"{utils.DATA_DIR}/sklearn_feature_names.joblib")
joblib.dump(ohe, f"{utils.MODELS_DIR}/sklearn_onehot_encoder.joblib")
joblib.dump(scaler, f"{utils.MODELS_DIR}/sklearn_scaler.joblib")

timing = {
    "encode_onehot_seconds": round(t_encode, 3),
    "scale_seconds": round(t_scale, 3),
    "split_seconds": round(t_split, 3),
    "total_preprocessing_seconds": round(t_total, 3),
    "n_rows": int(X.shape[0]), "n_features": int(X.shape[1]),
    "n_train": int(X_train.shape[0]), "n_test": int(X_test.shape[0]),
}
utils.save_json(timing, f"{utils.RESULTS_DIR}/02_sklearn_preprocessing_timing.json")
print(timing)

{'encode_onehot_seconds': 3.017, 'scale_seconds': 0.522, 'split_seconds': 1.058, 'total_preprocessing_seconds': 4.597, 'n_rows': 2260668, 'n_features': 72, 'n_train': 1808534, 'n_test': 452134}


**Nota metodológica:** siguiendo el orden explícito del enunciado, el
encoder y el scaler se ajustan (`fit`) sobre el dataset completo *antes*
del split, no solo sobre train. Con 2.26M filas y un split 80/20 el efecto
de esta simplificación sobre la media/desviación estándar usadas por
`StandardScaler` es marginal (el 20% de test apenas mueve esos estadísticos
a esta escala), pero en un dataset más pequeño lo metodológicamente más
estricto sería ajustar el `fit` solo sobre `X_train` y aplicar `transform`
a `X_test`.